In [1]:
import os
import subprocess
import time
import sys

def install_packages():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "streamlit", "pyngrok", "pyjwt", "watchdog"])

print("Installing required packages...")
install_packages()

from pyngrok import ngrok

os.makedirs(".streamlit", exist_ok=True)

config_toml = """
[theme]
base="dark"
primaryColor="#4F8BF9"
backgroundColor="#0E1117"
secondaryBackgroundColor="#262730"
textColor="#FAFAFA"
font="sans serif"
"""

with open(".streamlit/config.toml", "w") as f:
    f.write(config_toml)

print("Theme applied")
!service postgresql start
!sudo -u postgres psql -c "ALTER USER postgres PASSWORD 'postgres';"
!sudo -u postgres psql -c "CREATE DATABASE milestone1_users;"

Installing required packages...
Applied Dark Theme configuration.


In [6]:
import subprocess
import socket
import time

def wait_for_streamlit(port=8501, timeout=30):
    start = time.time()
    while time.time() - start < timeout:
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        if sock.connect_ex(('localhost', port)) == 0:
            sock.close()
            return True
        sock.close()
        time.sleep(1)
    return False

print("Enter Ngrok Token:")
authtoken = input()

ngrok.set_auth_token(authtoken)

os.system("pkill ngrok")
os.system("pkill streamlit")

process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

if wait_for_streamlit():
    url = ngrok.connect(8501).public_url
    print("🚀 App Running:", url)
else:
    print("Streamlit failed")

Enter Ngrok Token:
3AZEfiIIuIfMiGOXFV0YT3nkdd8_3Cgsp4jXvw2UAKGXCdnLs


KeyboardInterrupt: 

In [12]:
import streamlit as st
import jwt
import datetime
import bcrypt
import re

# ---------- CONFIG ----------
SECRET = "my_secret_key"
ALGO = "HS256"

# ---------- SESSION STORAGE ----------
if "users" not in st.session_state:
    st.session_state["users"] = {}

if "token" not in st.session_state:
    st.session_state["token"] = None

if "screen" not in st.session_state:
    st.session_state["screen"] = "login"

# ---------- JWT ----------
def generate_token(email, username):
    data = {
        "email": email,
        "username": username,
        "exp": datetime.datetime.utcnow() + datetime.timedelta(minutes=30)
    }
    return jwt.encode(data, SECRET, algorithm=ALGO)

def check_token(token):
    try:
        return jwt.decode(token, SECRET, algorithms=[ALGO])
    except:
        return None

# ---------- VALIDATION ----------
def check_email(e):
    return re.match(r'^[\w\.-]+@[\w\.-]+\.\w+$', e)

def check_pass(p):
    return len(p) >= 8 and p.isalnum()

# ---------- SIGNUP ----------
def signup():
    st.title("Signup Page")

    uname = st.text_input("Username")
    mail = st.text_input("Email")
    pwd = st.text_input("Password", type="password")
    cpwd = st.text_input("Confirm Password", type="password")

    ques = st.selectbox("Security Question", [
        "What is your pet name?",
        "What is your mother’s maiden name?",
        "What is your favorite teacher?"
    ])

    ans = st.text_input("Security Answer")

    if st.button("Register"):
        if not all([uname, mail, pwd, cpwd, ans]):
            st.error("All fields are required")
        elif not check_email(mail):
            st.error("Invalid email")
        elif not check_pass(pwd):
            st.error("Password must be 8+ alphanumeric")
        elif pwd != cpwd:
            st.error("Passwords do not match")
        elif mail in st.session_state["users"]:
            st.error("User already exists")
        else:
            hashed = bcrypt.hashpw(pwd.encode(), bcrypt.gensalt()).decode()

            st.session_state["users"][mail] = {
                "username": uname,
                "password": hashed,
                "question": ques,
                "answer": ans.lower()
            }

            st.success("Account created!")
            st.session_state["screen"] = "login"
            st.rerun()

    if st.button("Back to Login"):
        st.session_state["screen"] = "login"
        st.rerun()

# ---------- LOGIN ----------
def login():
    st.title("Login Page")

    mail = st.text_input("Email")
    pwd = st.text_input("Password", type="password")

    if st.button("Login"):
        user = st.session_state["users"].get(mail)

        if user:
            if bcrypt.checkpw(pwd.encode(), user["password"].encode()):
                st.session_state["token"] = generate_token(mail, user["username"])
                st.success("Login successful")
                st.rerun()
            else:
                st.error("Wrong password")
        else:
            st.error("User not found")

    col1, col2 = st.columns(2)

    if col1.button("Signup"):
        st.session_state["screen"] = "signup"
        st.rerun()

    if col2.button("Forgot Password"):
        st.session_state["screen"] = "forgot"
        st.rerun()

# ---------- FORGOT PASSWORD ----------
def forgot():
    st.title("Forgot Password")

    mail = st.text_input("Enter Email")

    if st.button("Check"):
        user = st.session_state["users"].get(mail)

        if user:
            st.session_state["mail"] = mail
            st.session_state["q"] = user["question"]
            st.session_state["a"] = user["answer"]
        else:
            st.error("Email not found")

    if "q" in st.session_state:
        st.info(st.session_state["q"])
        ans = st.text_input("Answer")

        if st.button("Verify"):
            if ans.lower() == st.session_state["a"]:
                st.session_state["allow"] = True
            else:
                st.error("Wrong answer")

    if st.session_state.get("allow"):
        newp = st.text_input("New Password", type="password")

        if st.button("Update Password"):
            if check_pass(newp):
                hashed = bcrypt.hashpw(newp.encode(), bcrypt.gensalt()).decode()
                st.session_state["users"][st.session_state["mail"]]["password"] = hashed

                st.success("Password updated!")
                st.session_state.clear()
                st.session_state["screen"] = "login"
                st.rerun()
            else:
                st.error("Invalid password")

# ---------- DASHBOARD ----------
def dashboard():
    data = check_token(st.session_state["token"])

    if not data:
        st.session_state["token"] = None
        st.rerun()

    st.title(f"Welcome {data['username']}")
    st.success("You are logged in")

    if st.button("Logout"):
        st.session_state.clear()
        st.session_state["screen"] = "login"
        st.rerun()

# ---------- ROUTER ----------
if st.session_state["token"]:
    dashboard()
else:
    if st.session_state["screen"] == "signup":
        signup()
    elif st.session_state["screen"] == "forgot":
        forgot()
    else:
        login()

2026-03-19 10:06:53.983 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-19 10:06:53.984 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-19 10:06:53.985 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-19 10:06:53.987 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-19 10:06:53.989 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-19 10:06:53.990 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-19 10:06:53.991 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-19 10:06:53.991 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [20]:
!pip install streamlit pyjwt bcrypt pyngrok

In [21]:
from pyngrok import ngrok
import subprocess

ngrok.set_auth_token("3AZEfiIIuIfMiGOXFV0YT3nkdd8_3Cgsp4jXvw2UAKGXCdnLs")

process = subprocess.Popen(["streamlit", "run", "app.py"])

url = ngrok.connect(8501).public_url
print("🚀 Open this:", url)

🚀 Open this: https://ungrateful-neoma-irresponsibly.ngrok-free.dev


In [22]:
from google.colab import files
files.download('app.py')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip install streamlit psycopg2-binary bcrypt pyjwt pyngrok watchdog

In [ ]:
# --- Wait for Streamlit to Start ---
def wait_for_streamlit(port=8501, timeout=30):
    start_time = time.time()
    while time.time() - start_time < timeout:
        try:
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            sock.settimeout(1)
            result = sock.connect_ex(('localhost', port))
            if result == 0:
                sock.close()
                return True
            sock.close()
        except Exception:
            pass
        time.sleep(1)
    return False
# --- Ngrok Setup ---
print("\nTo access the app, you need an Ngrok Authtoken.")
print("Get it from: https://dashboard.ngrok.com/get-started/your-authtoken")
authtoken = input("Enter your Ngrok Authtoken: ").strip()
if authtoken:
    ngrok.set_auth_token(authtoken)

    # Kill any existing ngrok process
    os.system("pkill ngrok")
    os.system("pkill streamlit")

    # Run Streamlit in the background FIRST
    print("Starting Streamlit...")
    # Using Subprocess.Popen to run in background
    # Redirecting output to /dev/null to keep cell clean
    process = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "localhost"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    # Wait for it to be ready
    if wait_for_streamlit():
        print("Streamlit is active! Connecting Ngrok...")
        # Open a tunnel to the streamlit port 8501
        try:
            public_url = ngrok.connect(8501).public_url
            print(f"\n🚀 Streamlit App is running!")
            print(f"👉 Public URL: {public_url}")
            print("\n(Click the URL above to open the app)")

            # Keep main thread alive
            try:
                # Keep checking if process is alive
                while process.poll() is None:
                    time.sleep(1)
            except KeyboardInterrupt:
                print("Stopping...")
                ngrok.disconnect(public_url)
                process.terminate()
        except Exception as e:
            print(f"Ngrok connection failed: {e}")
            process.terminate()
    else:
        print("Error: Streamlit failed to start in time.")
        process.terminate()
else:
    print("Ngrok Authtoken is required to expose the app publicly.")

In [19]:
%%writefile app.py
import streamlit as st
import jwt
import datetime
import bcrypt
import re

# ---------- CONFIG ----------
SECRET = "my_secret_key"
ALGO = "HS256"

# ---------- SESSION STORAGE ----------
if "users" not in st.session_state:
    st.session_state["users"] = {}

if "token" not in st.session_state:
    st.session_state["token"] = None

if "screen" not in st.session_state:
    st.session_state["screen"] = "login"

# ---------- JWT ----------
def generate_token(email, username):
    data = {
        "email": email,
        "username": username,
        "exp": datetime.datetime.utcnow() + datetime.timedelta(minutes=30)
    }
    return jwt.encode(data, SECRET, algorithm=ALGO)

def check_token(token):
    try:
        return jwt.decode(token, SECRET, algorithms=[ALGO])
    except:
        return None

# ---------- VALIDATION ----------
def check_email(e):
    return re.match(r'^[\w\.-]+@[\w\.-]+\.\w+$', e)

def check_pass(p):
    return len(p) >= 8 and p.isalnum()

# ---------- SIGNUP ----------
def signup():
    st.title("Signup Page")

    uname = st.text_input("Username")
    mail = st.text_input("Email")
    pwd = st.text_input("Password", type="password")
    cpwd = st.text_input("Confirm Password", type="password")

    ques = st.selectbox("Security Question", [
    "What is your pet name?",
    "What is your mother’s maiden name?",
    "What is your favorite teacher?",
    "What is your favorite food?",
    "What is your birth city?"
])

    ans = st.text_input("Security Answer")

    if st.button("Register"):
        if not all([uname, mail, pwd, cpwd, ans]):
            st.error("All fields are required")
        elif not check_email(mail):
            st.error("Invalid email")
        elif not check_pass(pwd):
            st.error("Password must be 8+ alphanumeric")
        elif pwd != cpwd:
            st.error("Passwords do not match")
        elif mail in st.session_state["users"]:
            st.error("User already exists")
        else:
            hashed = bcrypt.hashpw(pwd.encode(), bcrypt.gensalt()).decode()

            st.session_state["users"][mail] = {
                "username": uname,
                "password": hashed,
                "question": ques,
                "answer": ans.lower()
            }

            st.success("Account created!")
            st.session_state["screen"] = "login"
            st.rerun()

    if st.button("Back to Login"):
        st.session_state["screen"] = "login"
        st.rerun()

# ---------- LOGIN ----------
def login():
    st.title("Login Page")

    mail = st.text_input("Email")
    pwd = st.text_input("Password", type="password")

    if st.button("Login"):
        user = st.session_state["users"].get(mail)

        if user:
            if bcrypt.checkpw(pwd.encode(), user["password"].encode()):
                st.session_state["token"] = generate_token(mail, user["username"])
                st.success("Login successful")
                st.rerun()
            else:
                st.error("Wrong password")
        else:
            st.error("User not found")

    col1, col2 = st.columns(2)

    if col1.button("Signup"):
        st.session_state["screen"] = "signup"
        st.rerun()

    if col2.button("Forgot Password"):
        st.session_state["screen"] = "forgot"
        st.rerun()

# ---------- FORGOT PASSWORD ----------
def forgot():
    st.title("Forgot Password")

    mail = st.text_input("Enter Email")

    if st.button("Check"):
        user = st.session_state["users"].get(mail)

        if user:
            st.session_state["mail"] = mail
            st.session_state["q"] = user["question"]
            st.session_state["a"] = user["answer"]
        else:
            st.error("Email not found")

    if "q" in st.session_state:
        st.info(st.session_state["q"])
        ans = st.text_input("Answer")

        if st.button("Verify"):
            if ans.lower() == st.session_state["a"]:
                st.session_state["allow"] = True
            else:
                st.error("Wrong answer")

    if st.session_state.get("allow"):
        newp = st.text_input("New Password", type="password")

        if st.button("Update Password"):
            if check_pass(newp):
                hashed = bcrypt.hashpw(newp.encode(), bcrypt.gensalt()).decode()
                st.session_state["users"][st.session_state["mail"]]["password"] = hashed

                st.success("Password updated!")
                st.session_state.clear()
                st.session_state["screen"] = "login"
                st.rerun()
            else:
                st.error("Invalid password")

# ---------- DASHBOARD ----------
def dashboard():
    data = check_token(st.session_state["token"])

    if not data:
        st.session_state["token"] = None
        st.rerun()

    st.title(f"Welcome {data['username']}")
    st.success("You are logged in")

    if st.button("Logout"):
        st.session_state.clear()
        st.session_state["screen"] = "login"
        st.rerun()

# ---------- ROUTER ----------
if st.session_state["token"]:
    dashboard()
else:
    if st.session_state["screen"] == "signup":
        signup()
    elif st.session_state["screen"] == "forgot":
        forgot()
    else:
        login()

Overwriting app.py


In [15]:
!pip install streamlit pyjwt bcrypt pyngrok